In [1]:
import torch
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

device: mps


In [2]:
model_name = "sentence-transformers/all-distilroberta-v1"

model = SentenceTransformer(model_name, device=str(device))
print(model_name)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: sentence-transformers/all-distilroberta-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


sentence-transformers/all-distilroberta-v1


In [3]:
ds = load_dataset("glue", "mrpc", split="validation")
print(ds)
print(ds[0])

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}


In [4]:
sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

print("num_examples:", len(y_true))
print("positive_rate:", y_true.mean())

num_examples: 408
positive_rate: 0.6838235294117647


In [5]:
batch_size = 128

emb1 = model.encode(
    sent1,
    batch_size=batch_size,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

emb2 = model.encode(
    sent2,
    batch_size=batch_size,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

print("emb1 shape:", tuple(emb1.shape))
print("emb2 shape:", tuple(emb2.shape))

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

emb1 shape: (408, 768)
emb2 shape: (408, 768)


In [6]:
scores = torch.sum(emb1 * emb2, dim=1)
threshold = 0.72
y_pred = (scores >= threshold).to(torch.int64).cpu().numpy()
scores_np = scores.cpu().numpy()

print("done")
print("threshold:", threshold)
print("score_range:", (float(scores_np.min()), float(scores_np.max())))

done
threshold: 0.72
score_range: (0.22469396889209747, 0.9964145421981812)


In [7]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))

{'accuracy': 0.7230392156862745, 'f1': 0.8055077452667814}
                precision    recall  f1-score   support

not_paraphrase       0.58      0.47      0.52       129
    paraphrase       0.77      0.84      0.81       279

      accuracy                           0.72       408
     macro avg       0.68      0.66      0.66       408
  weighted avg       0.71      0.72      0.71       408



In [8]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("score:", float(scores_np[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))

sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
score: 0.9106887578964233
true: 1 pred: 1
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
score: 0.35538217425346375
true: 0 pred: 0
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
score: 0.7522245049476624
true: 0 pred: 1
sentence1: The AFL-CIO is waiting until October to decide if it will endorse a candidate .
sentence2: The AFL-CIO announced Wednesday that it will de

In [9]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("score:", float(scores_np[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))

num_errors: 113
idx: 2
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
score: 0.7522245049476624
true: 0 pred: 1
idx: 5
sentence1: Wal-Mart said it would check all of its million-plus domestic workers to ensure they were legally employed .
sentence2: It has also said it would review all of its domestic employees more than 1 million to ensure they have legal status .
score: 0.6570861339569092
true: 1 pred: 0
idx: 6
sentence1: While dioxin levels in the environment were up last year , they have dropped by 75 percent since the 1970s , said Caswell .
sentence2: The Institute said dioxin levels in the environment have fallen by as much as 76 percent since the 1970s .
score: 0.8033972978591919
true: 0 pred: 1
idx: 11
sentence1: " Sanitation is poor ... there could be

In [10]:
summary = {
    "dataset": "glue/mrpc",
    "split": "validation",
    "model": model_name,
    "device": str(device),
    "threshold": float(threshold),
    "num_examples": len(ds),
    "accuracy": float(acc),
    "f1": float(f1),
}
summary

{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'sentence-transformers/all-distilroberta-v1',
 'device': 'mps',
 'threshold': 0.72,
 'num_examples': 408,
 'accuracy': 0.7230392156862745,
 'f1': 0.8055077452667814}